# Code was run on Colab Pro

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import collections
import torch.optim as optim
from torch.optim import Optimizer
import time
import matplotlib.pyplot as plt

from AdamW          import AdamW
from utils          import utility, misreportUtility, misreportOptimization, trueUtility, loss
from networks       import AdditiveMechanism, Misreports,AllocationNet,PaymentNet
from restrictedAdam import Adam 
from networks import MixedWrapper

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.cuda.set_device(2)

# Set Random Seed 

In [3]:
# Initializing seeds
torch.manual_seed(5)
np.random.seed(5)

# Testing Function

In [4]:
def test(nBatch, nbrInit, R, gamma=0.001, minimum=0, maximum=1):
    """
    Evaluate both the pure neural network (Net) and the mixed mechanism (Mixed).
    - Keep the original misreportUtility / utility / loss implementation.
    - Disable straight-through for mixed evaluation (hard selection).
    """

    # Comment removed.

    reserve=0.5
    true = np.random.rand(nBatch, nAgent, nObject)
    localMisreports     = np.random.rand(nBatch, nbrInit, nAgent, nObject)
    batchMisreports     = torch.tensor(localMisreports).float().to(device)
    batchTrueValuations = torch.tensor(true).float().to(device)
    batchMisreports.requires_grad = True

    def _eval_once(mech_callable):
        # Misreport optimization followed by regret/payment/loss evaluation.
        opt = Adam([batchMisreports], lr=gamma)
        for k in range(R):
            advU = misreportUtility(mech_callable, batchTrueValuations, batchMisreports)
            los  = -1*torch.mean(advU).to(device)
            los.backward()
            opt.step(restricted=True, min=minimum, max=maximum)
            opt.zero_grad()

        misReportUtilityMax  = torch.max(advU, dim=1)[0]
        allocation, payment = mech_callable(batchTrueValuations)
        regret = F.relu(misReportUtilityMax - utility(batchTrueValuations, allocation, payment))
        mregret = torch.sum(torch.mean(regret, dim=0)).to(device)
        mregret = float(mregret.detach().cpu().numpy())
        with torch.no_grad():
            l, rMean, p = loss(payment, regret)
        return mregret, float(p.detach().cpu().numpy()), float((-l).detach().cpu().numpy())**2

    # ----- Define two mechanism callables -----
    # Pure neural network
    if hasattr(mechanism, "base"):   # Comment removed.

        def mech_net(X):  # Pure neural network
            return mechanism.base(X)
    else:
        def mech_net(X):  # Use the mechanism itself if it is not wrapped.
            return mechanism(X)

    # Mixed mechanism with hard profile-wise selection.
    st_backup = getattr(mechanism, "st", None)
    if hasattr(mechanism, "st"):
        mechanism.st = False
    def mech_mix(X):
        return mechanism(X)
    # --------------------

    # ----- Evaluate both mechanisms -----
    net_reg, net_pay, net_optrev   = _eval_once(mech_net)
    mix_reg, mix_pay, mix_optrev   = _eval_once(mech_mix)

    # Restore straight-through state if needed.
    if hasattr(mechanism, "st"):
        mechanism.st = st_backup if st_backup is not None else True

    # Print both results.
    print(f"[NET  ] regret={net_reg:.5f}  avg/bidder={net_reg/nAgent:.5f}  optRev={net_optrev:.3f}  payment={net_pay:.3f}")
    print(f"[MIXED] regret={mix_reg:.5f}  avg/bidder={mix_reg/nAgent:.5f}  optRev={mix_optrev:.3f}  payment={mix_pay:.3f}")
    testRegret.append(net_reg)
    testPayment.append(net_pay)
    testOptimal.append(net_optrev)

    # Return a dictionary for optional logging.
    return {
        "net":   {"regret": net_reg, "payment": net_pay, "optrev": net_optrev},
        "mixed": {"regret": mix_reg, "payment": mix_pay, "optrev": mix_optrev},
    }

    
def finaltest(nBatch, nbrInit, R, gamma=0.001, minimum=0, maximum=1):
    
    """ This function computes the regret and payment of mechanism on a test set of size nBatch
        The optimal misreport is computed by optimizing the utility function (not by using the Misreport network)
        for R gradient steps (of stepsize gamma) and starting from nbrInit initialization, we only keep the best misreport
        To compute the regret we evaluate the mechanism at the misreport and compare to the valuation
        minimum and maximum indicate the range of the valuations
    """
    
    true = np.random.rand(nBatch,nAgent,nObject)

    # Final evaluation uses the pure neural student, not the Stage III mixed wrapper.
    eval_mechanism = mechanism.base if hasattr(mechanism, "base") else mechanism
    eval_mechanism.eval()

   
    with torch.no_grad():
        batchTrue = torch.tensor(true).float().to(device)
        allocation, payment = eval_mechanism(batchTrue)
        batchTrueValuations = torch.tensor(true).float().to(device).unsqueeze(1).repeat(1,1000, 1, 1)
        localMisreports     =  np.expand_dims(true,1).repeat(1000,axis=1)
        max_u = torch.tensor(np.zeros((nBatch,nAgent,nObject))).float().to(device)
        best_misreport_values = torch.zeros(nBatch, nAgent, nObject).to(device)
        for l in range(nAgent):
            for i in range(nObject):
                localMisreports     = np.expand_dims(true,1).repeat(1000,axis=1)
                localMisreports[:,:,l,i]=0
                for k in range(nBatch):
                    for j in range(999):
                        localMisreports[k,j+1,l,i]=  localMisreports[k,j,l,i]+0.001

                batchMisreports     = torch.tensor(localMisreports).float().to(device)
            
                a_m,p_m = eval_mechanism(batchMisreports.reshape(-1,nAgent,nObject))
                utility1 = (a_m.reshape(nBatch,1000,nAgent,nObject)*batchTrueValuations).sum(dim=3) - p_m.reshape(nBatch,1000,nAgent)
                
                
                cur_u = torch.max(utility1,dim=1)[0][:,l]-utility(batchTrue, allocation, payment)[:,l]
                max_u[:,l,i] = cur_u
                best_idx = torch.argmax(utility1[:,:,l], dim=1)

                # Extract the selected misreport value.
                best_values = batchMisreports[
                    torch.arange(nBatch).to(device),
                    best_idx,
                            l,
                    i
                ]

                best_misreport_values[:,l,i] = best_values
        
        regret = F.relu(torch.sum(torch.max(max_u,axis=2)[0])/nBatch)
        mregret= float(regret.cpu().detach().numpy())

        regret = F.relu(torch.sum(torch.max(max_u,axis=2)[0])/nBatch)
        mregret= float(regret.cpu().detach().numpy())
        with torch.no_grad():
            l,rMean,p = loss(payment, regret)

    


    batchTrueValuations = torch.tensor(true, dtype=torch.float32, device=device)  # [B,A,M]
    full_best = best_misreport_values                                            # [B,A,M]

    B = nBatch
    A = nAgent
    M = nObject
    m = nObject

    mis_list = []

    # Candidate 1: use the best value for all items.
    mis_full = full_best                                     # [B,A,M]
    mis_list.append(mis_full)
    # Candidate 2: change one item at a time and keep the others truthful.
    for i_obj in range(m):
        mis_i = batchTrueValuations.clone()            # [B, nAgent, nObject]
        mis_i[:, :, i_obj] = full_best[:, :, i_obj]    # Change only item i_obj.
        mis_list.append(mis_i)

    # Candidate 3: for each bidder, change only the item with the largest regret.
    best_item_idx = torch.argmax(max_u, dim=2)               # [B,A]
    mis_bestitem = batchTrueValuations.clone()               # [B,A,M]
    b_idx = torch.arange(B, device=device)
    for l in range(A):
        idx_l = best_item_idx[:, l]                          # [B]
        mis_bestitem[b_idx, l, idx_l] = full_best[b_idx, l, idx_l]
    mis_list.append(mis_bestitem)

# Optional global random misreport initializations.
    num_global_random = 1
    for _ in range(num_global_random):
        mis_rand = torch.rand(B, A, M, device=device)
        mis_list.append(mis_rand)

# Optional local perturbations around truthful bids.
    num_local_around_truth = 1
    noise_scale = 0.2
    for _ in range(num_local_around_truth):
        noise = torch.randn(B, A, M, device=device) * noise_scale
        mis_loc = batchTrueValuations + noise
        mis_loc = torch.clamp(mis_loc, 0.0, 1.0)
        mis_list.append(mis_loc)

# Optional local perturbations around the best misreports.
    num_local_around_best = 1
    noise_scale_best = 0.2
    for _ in range(num_local_around_best):
        noise = torch.randn(B, A, M, device=device) * noise_scale_best
        mis_loc_best = full_best + noise
        mis_loc_best = torch.clamp(mis_loc_best, 0.0, 1.0)
        mis_list.append(mis_loc_best)

    # ===== Stack all candidate initializations =====
    localMisreports = torch.stack(mis_list, dim=1)           # [B, K, A, M]
    batchMisreports = localMisreports.clone().detach()
    batchMisreports.requires_grad = True


    opt = Adam([batchMisreports], lr=gamma)
    
    for k in range(R):
        advU         = misreportUtility(eval_mechanism,batchTrueValuations,batchMisreports)
        los          =  -1*torch.mean(advU).to(device)
        los.backward()
        opt.step(restricted= True, min=minimum, max=maximum)
        opt.zero_grad()
    
    misReportUtilityMax  = torch.max(advU, dim =1)[0]
    eval_mechanism.zero_grad()
    allocation, payment = eval_mechanism(batchTrueValuations)
    regret = F.relu(misReportUtilityMax -utility(batchTrueValuations, allocation, payment))
    mregret= torch.sum(torch.mean(regret, dim=0)).to(device)
    mregret= float(mregret.cpu().detach().numpy())

    with torch.no_grad():
        l,rMean,p = loss(payment, regret)

    testRegret.append(mregret)
    testPayment.append(float(p.detach().cpu().numpy() ))
    testOptimal.append(float((-l).detach().cpu().numpy())**2)
    
    print("Total regret: ",'{0:.5f}'.format(mregret), "Average regret per bidder: ",'{0:.5f}'.format(mregret/nAgent), " Optimal Revenue: ",'{0:.3f}'.format(float((-l).detach().cpu().numpy())**2), " payment: ",'{0:.3f}'.format(float(p.detach().cpu().numpy() )))

# Initializing Networks

In [5]:
nAgent   = 3
nObject  = 10

# Parameters for the mechanism (payment and allocation network)
nLayersAllocation   = 7
nLayersPayment      = 7
widthAllocation     = 100
widthPayment        = 100

# Parameters for the misreport network
nLayersMisreport    = 7
widthMisreport      = 100

gamma              = 0.001 
testBatch          = 10000

nExperiments       = 200000
batchSize          = 500
nbrBatches         = int(nExperiments/batchSize)


mechanism_base            = AdditiveMechanism(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
mechanism_base= torch.load("310-stage2.pt")
mechanism = MixedWrapper(mechanism_base, reserve=0.5, straight_through=True).to(device)
mechanism.train() 
optimizerMechanism   = AdamW(mechanism_base .parameters(), lr=0.0001)

misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
optimizerMisreport   = AdamW(misreport.parameters(), lr=0.001)

In [6]:
testRegret    = []
testMaxRegret = []
testPayment   = []
testOptimal   = []
testTime      = []
testIteration = [0]

# range of valuations
minimum            = 0
maximum            = 1

# Training

In [7]:
duration   = 0
R          = 100

i=0

print("Initial Test")
test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

for t in range(1,60*nbrBatches+1):
    
    # Reinitialize Misreport network periodically at the beginning of training
    if (t%(2*nbrBatches) ==1):
      if   t< 20*nbrBatches+2 :
    
        misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
        optimizerMisreport   = AdamW(misreport.parameters(), lr=0.001)

    batchTrueValuations = torch.tensor(np.random.rand(batchSize,nAgent,nObject)).float().to(device)
    
    # Optimize Misreport Network for R steps
    for k in range(R):
  
        misreports          = misreport(batchTrueValuations).unsqueeze(1)
        mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)
        mLoss               = torch.sum(torch.mean(-mUtility,dim=0))

        optimizerMisreport.zero_grad()
        mLoss.backward()
        optimizerMisreport.step()

    
    # Optimize Mechanism network for one step
    misreports          = misreport(batchTrueValuations).unsqueeze(1)
    mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)

    allocation, payment = mechanism(batchTrueValuations)

    regret     = 1.5*F.relu(mUtility -utility(batchTrueValuations, allocation, payment))
    l,rMean,p = loss(payment, regret)
        
    optimizerMechanism.zero_grad()

    l.backward()

    optimizerMechanism.step()
    
    # Test mechanism periodically
    if t % (2*nbrBatches)==0 :
        print("Batch: ", 2*int(t/(2*nbrBatches)))
        testTime.append(duration)
        testIteration.append(t/nbrBatches)
        test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

Initial Test


/home/wkw/ysy/hunhe/restrictedAdam.py:103: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha) (Triggered internally at  ../torch/csrc/utils/python_arg_parser.cpp:1050.)
  exp_avg.mul_(beta1).add_(1 - beta1, grad)


[NET  ] regret=0.00506  avg/bidder=0.00169  optRev=5.480  payment=5.842
[MIXED] regret=0.25636  avg/bidder=0.08545  optRev=2.756  payment=5.870
Batch:  2
[NET  ] regret=0.00816  avg/bidder=0.00272  optRev=5.431  payment=5.900
[MIXED] regret=0.15983  avg/bidder=0.05328  optRev=3.497  payment=5.903
Batch:  4
[NET  ] regret=0.00955  avg/bidder=0.00318  optRev=5.386  payment=5.896
[MIXED] regret=0.17713  avg/bidder=0.05904  optRev=3.362  payment=5.913
Batch:  6
[NET  ] regret=0.01252  avg/bidder=0.00417  optRev=5.385  payment=5.978
[MIXED] regret=0.21971  avg/bidder=0.07324  optRev=3.095  payment=5.991
Batch:  8
[NET  ] regret=0.01195  avg/bidder=0.00398  optRev=5.252  payment=5.822
[MIXED] regret=0.20390  avg/bidder=0.06797  optRev=3.101  payment=5.840
Batch:  10
[NET  ] regret=0.01393  avg/bidder=0.00464  optRev=5.382  payment=6.011
[MIXED] regret=0.23298  avg/bidder=0.07766  optRev=3.042  payment=6.051
Batch:  12
[NET  ] regret=0.01431  avg/bidder=0.00477  optRev=5.043  payment=5.663
[M

# Testing

In [8]:
for i in range(200):
    finaltest(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

Total regret:  0.01046 Average regret per bidder:  0.00349  Optimal Revenue:  5.320  payment:  5.852
Total regret:  0.00953 Average regret per bidder:  0.00318  Optimal Revenue:  5.341  payment:  5.848
Total regret:  0.01080 Average regret per bidder:  0.00360  Optimal Revenue:  5.313  payment:  5.855
Total regret:  0.01026 Average regret per bidder:  0.00342  Optimal Revenue:  5.322  payment:  5.849
Total regret:  0.01019 Average regret per bidder:  0.00340  Optimal Revenue:  5.189  payment:  5.708
Total regret:  0.01078 Average regret per bidder:  0.00359  Optimal Revenue:  5.235  payment:  5.773
Total regret:  0.00979 Average regret per bidder:  0.00326  Optimal Revenue:  5.234  payment:  5.743
Total regret:  0.01093 Average regret per bidder:  0.00364  Optimal Revenue:  5.292  payment:  5.837
Total regret:  0.00945 Average regret per bidder:  0.00315  Optimal Revenue:  5.296  payment:  5.799
Total regret:  0.01105 Average regret per bidder:  0.00368  Optimal Revenue:  5.228  paymen

Total regret:  0.01160 Average regret per bidder:  0.00387  Optimal Revenue:  5.237  payment:  5.798
Total regret:  0.00971 Average regret per bidder:  0.00324  Optimal Revenue:  5.291  payment:  5.801
Total regret:  0.00992 Average regret per bidder:  0.00331  Optimal Revenue:  5.263  payment:  5.777
Total regret:  0.00987 Average regret per bidder:  0.00329  Optimal Revenue:  5.328  payment:  5.844
Total regret:  0.01056 Average regret per bidder:  0.00352  Optimal Revenue:  5.294  payment:  5.828
Total regret:  0.01066 Average regret per bidder:  0.00355  Optimal Revenue:  5.236  payment:  5.770
Total regret:  0.01001 Average regret per bidder:  0.00334  Optimal Revenue:  5.105  payment:  5.615
Total regret:  0.00997 Average regret per bidder:  0.00332  Optimal Revenue:  5.234  payment:  5.749
Total regret:  0.01055 Average regret per bidder:  0.00352  Optimal Revenue:  5.240  payment:  5.772
Total regret:  0.00903 Average regret per bidder:  0.00301  Optimal Revenue:  5.243  paymen

Total regret:  0.01191 Average regret per bidder:  0.00397  Optimal Revenue:  5.084  payment:  5.645
Total regret:  0.00962 Average regret per bidder:  0.00321  Optimal Revenue:  5.302  payment:  5.810
Total regret:  0.00966 Average regret per bidder:  0.00322  Optimal Revenue:  5.172  payment:  5.675
Total regret:  0.00972 Average regret per bidder:  0.00324  Optimal Revenue:  5.345  payment:  5.858
Total regret:  0.01150 Average regret per bidder:  0.00383  Optimal Revenue:  5.306  payment:  5.867
Total regret:  0.01128 Average regret per bidder:  0.00376  Optimal Revenue:  5.229  payment:  5.780
Total regret:  0.01011 Average regret per bidder:  0.00337  Optimal Revenue:  5.281  payment:  5.802
Total regret:  0.01080 Average regret per bidder:  0.00360  Optimal Revenue:  5.161  payment:  5.695
Total regret:  0.00960 Average regret per bidder:  0.00320  Optimal Revenue:  5.221  payment:  5.724
Total regret:  0.00951 Average regret per bidder:  0.00317  Optimal Revenue:  5.266  paymen

In [9]:
totalregret = np.mean(np.array(testRegret[-200:]))
revenue     = np.mean(np.array(testPayment[-200:]))
print("Final Result")
print("Total Regret = ", '{0:.5f}'.format(totalregret), "Average regret per bidder: ",'{0:.5f}'.format(totalregret/nAgent), " Optimal Revenue: ",'{0:.3f}'.format(float(np.sqrt(revenue)-np.sqrt(totalregret))**2), " payment: ",'{0:.3f}'.format(revenue))

Final Result
Total Regret =  0.01036 Average regret per bidder:  0.00345  Optimal Revenue:  5.302  payment:  5.781


In [10]:
stdregret = np.std(np.array(testRegret[-200:]))
stdrevenue= np.std(np.array(testPayment[-200:]))
print("std Regret = ", '{0:.5f}'.format(stdregret), "std regret per bidder: ",'{0:.5f}'.format(stdregret/nAgent), " std payment: ",'{0:.3f}'.format(stdrevenue))

std Regret =  0.00087 std regret per bidder:  0.00029  std payment:  0.082


In [12]:
torch.save(mechanism,'310stage3.pt')